# 🗂️ MaintiDoc — Assistant Agentique Multimodal de Maintenance SAN
## Notebook d'Orchestration : Ingestion Modulaire → Qdrant Vector Store → Fusion RRF → Qwen3.8 VLM

> **Architecture Industrielle** : Ce notebook ne contient plus de logique bas niveau monolithique. Il orchestre les modules testés et typés du package `src/maintidoc`.

### Pipeline Fonctionnel :
1. **Ingestion & Détection Structurée** : Extraction PyMuPDF traçable (textes, tableaux Markdown, schémas techniques `FIG-01` à `FIG-05`).
2. **Stockage Vectoriel Industriel (Sprint 1)** : Indexation dans **Qdrant** avec séparation des collections texte (1024-d, BGE-M3) et image (512-d, CLIP).
3. **Recherche Hybride & Fusion RRF** : Algorithme *Reciprocal Rank Fusion* pour aligner texte et schémas au niveau page.
4. **Génération Multimodale Structurée** : Inférence **Qwen** via Ollama avec contrainte de schéma JSON strict et sans hallucination.
5. **Contrôle Déterministe** : Audit de provenance vérifiant l'existence exacte de chaque citation.

### 0. Initialisation de l'Environnement et Import du Package `src/maintidoc`

In [1]:
import sys
from pathlib import Path

# Ajout du package modulaire src au path Python
SRC_PATH = Path.cwd() / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import pandas as pd
from PIL import Image
from IPython.display import display

from maintidoc.config import DEFAULT_PDF_PATH, DOC_ID
from maintidoc.ingestion import PDFExtractor
from maintidoc.indexing import EmbeddingsManager, QdrantStore
from maintidoc.retrieval import fuse_rrf
from maintidoc.generation import ask_maintidoc_vlm, validate_citations

print(f"✅ Environnement MaintiDoc initialisé. Document cible : {DEFAULT_PDF_PATH.name}")

ModuleNotFoundError: No module named 'qdrant_client'

### 1. Inventaire et Contrôle d'Intégrité des Pages du Manuel SAN

In [ ]:
# Initialisation de l'extracteur modulaire
extractor = PDFExtractor(pdf_path=DEFAULT_PDF_PATH)
inventory = extractor.get_inventory()
df_inventory = pd.DataFrame(inventory)
display(df_inventory)

assert len(inventory) == 9, "Erreur : Le document de référence doit comporter exactement 9 pages."
print(f"✅ Contrôle d'inventaire validé : {len(inventory)} pages détectées avec leurs tableaux et images.")

### 2. Ingestion & Extraction Granulaire (Textes, Tableaux, Schémas)

In [ ]:
# Extraction complète et typée d'après nos modèles Pydantic
text_records, table_records, image_records = extractor.extract_all()

print("📊 Données extraites avec succès :")
print(f"  - Paragraphes de texte : {len(text_records)}")
print(f"  - Tableaux Markdown     : {len(table_records)}")
print(f"  - Images & Schémas      : {len(image_records)}")

schema_records = [r for r in image_records if r.modality == "schema"]
assert len(schema_records) >= 5, "Erreur : Les 5 schémas techniques (FIG-01 à FIG-05) doivent être présents."
print(f"✅ {len(schema_records)} schémas techniques identifiés et enregistrés dans data/processed/.")

### 3. Inspection des Schémas Techniques Extraits (FIG-01 à FIG-05)

In [ ]:
for rec in schema_records:
    print(f"📐 {rec.figure_id} | Page {rec.page} | Fichier : {rec.image_path}")
    with Image.open(rec.image_path) as im:
        thumb = im.copy()
        thumb.thumbnail((650, 320))
        display(thumb)

### 4. Stockage Vectoriel Industriel : Qdrant Vector Store

Nous remplaçons les anciens index éphémères (`npz` et mémoire volatile) par **Qdrant**.
- Collection `maintidoc_text` : 1024 dimensions (vecteurs BGE-M3 pour textes et tableaux).
- Collection `maintidoc_visual` : 512 dimensions (vecteurs CLIP ViT-B/32 pour schémas techniques).

In [ ]:
# Initialisation du gestionnaire vectoriel et des modèles d'encodage
embeddings_mgr = EmbeddingsManager(device="cpu")
qdrant_store = QdrantStore()

# Création des collections vectorielles isolées
qdrant_store.init_collections(force_recreate=True)
mode_info = "Conteneur Docker actif" if qdrant_store.is_connected else "Mode Local / In-Memory"
print(f"✅ Qdrant prêt ({mode_info}). Collections 'maintidoc_text' & 'maintidoc_visual' créées.")

# Indexation
print("⏳ Indexation des textes et tableaux dans Qdrant (BGE-M3)...")
qdrant_store.index_texts_and_tables(text_records, table_records, embeddings_mgr)

print("⏳ Indexation des schémas techniques dans Qdrant (CLIP)...)")
qdrant_store.index_images(schema_records, embeddings_mgr)
print("✅ Indexation vectorielle industrielle terminée avec succès !")

### 5. Recherche Hybride Multimodale (Texte & Image)

In [ ]:
QUESTION = "Que signifie le voyant orange sur le SAN-LAB 2000 ?"
VISUAL_QUERY_EN = "storage SAN front control panel orange warning indicator status LEDs"

# 1. Recherche sémantique texte / tableaux dans Qdrant
text_hits = qdrant_store.search_text(QUESTION, embeddings_mgr, top_k=6)

# 2. Recherche visuelle dans les schémas techniques via CLIP
visual_hits = qdrant_store.search_visual(VISUAL_QUERY_EN, embeddings_mgr, top_k=5)

print("🔍 Top Hits Textuels :", [(h.element_id, f"Page {h.page}", f"Score: {round(h.score, 3)}") for h in text_hits])
print("🖼️ Top Hits Visuels  :", [(h.metadata.get("figure_id"), f"Page {h.page}", f"Score: {round(h.score, 3)}") for h in visual_hits])

### 6. Fusion des Résultats par Reciprocal Rank Fusion (RRF)

Formule mathématique : $\mathrm{RRF}(d) = \sum_{m \in \{\text{texte}, \text{visuel}\}} \frac{1}{k + \text{rank}_m(d)}$ (avec $k=60$).

In [ ]:
# Préparation des listes de candidats pour la fusion au niveau page
text_candidates = [{"page": h.page, "rank": h.rank, "element_id": h.element_id} for h in text_hits]
visual_candidates = [{"page": h.page, "rank": h.rank, "figure_id": h.metadata.get("figure_id")} for h in visual_hits]

fusion_results = fuse_rrf(text_candidates, visual_candidates, k=60)
df_rrf = pd.DataFrame([r.model_dump() for r in fusion_results])
display(df_rrf)

top_pages = [r.page for r in fusion_results[:3]]
print(f"🎯 Pages prioritaires retenues par fusion RRF : {top_pages}")

### 7. Constitution du Dossier de Preuves Borné

In [ ]:
# Sélection stricte des preuves rattachées aux pages élues par RRF
selected_texts = []
for h in text_hits:
    if h.page in top_pages:
        selected_texts.append({
            "document_id": DOC_ID,
            "page": h.page,
            "element_id": h.element_id,
            "excerpt": h.metadata.get("content", "")[:2200]
        })

selected_images = []
for h in visual_hits:
    if h.page in top_pages and len(selected_images) < 1:
        selected_images.append({
            "document_id": DOC_ID,
            "page": h.page,
            "element_id": h.element_id,
            "figure_id": h.metadata.get("figure_id"),
            "image_path": h.metadata.get("image_path")
        })

allowed_citations = {
    f"{h['document_id']}:p{h['page']}:{h['element_id']}": h
    for h in selected_texts + selected_images
}

print(f"📋 Preuves retenues : {len(selected_texts)} textes/tableaux, {len(selected_images)} schéma(s).")
print("📌 Références formelles autorisées pour le modèle :", list(allowed_citations.keys()))

### 8. Inférence Multimodale via Ollama (Qwen JSON Strict)

In [ ]:
import json

try:
    vlm_response = ask_maintidoc_vlm(
        question=QUESTION,
        texts=selected_texts,
        images=selected_images,
    )
    print("🤖 Réponse JSON structurée de MaintiDoc :\n")
    print(json.dumps(vlm_response, ensure_ascii=False, indent=2))
except Exception as exc:
    print(f"ℹ️ Serveur Ollama indisponible ou conteneur éteint ({exc}).")
    print("Pour démarrer Ollama : lancez 'docker compose up -d' dans votre terminal.")

### 9. Contrôle Déterministe des Citations & Audit de Zéro-Hallucination

In [ ]:
if 'vlm_response' in locals() and isinstance(vlm_response, dict):
    issues = validate_citations(vlm_response, allowed_citations)
    if issues:
        print("⚠️ Citations invalides ou non vérifiables détectées :", issues)
    else:
        print("✅ Audit de provenance validé avec succès : 100% des citations proviennent de sources réelles !")
else:
    print("ℹ️ Exécutez la cellule précédente avec Ollama actif pour auditer la réponse.")